# Figure 6 only — SHAP on the fold-10 model (PTB-XL 543-feature pipeline)

Cells 1–4 are the unchanged setup cells from the v2.4 Revision notebook (Drive mount, CONFIG/auto-detect of the feature file, helper functions). Cell 5 trains the fold-10 model, computes exact TreeExplainer SHAP, and saves **Figure6_shap.png** plus `shap_fold10.npy`, `X_fold10.csv`, `shap_global_fold10.csv` to `_analysis_checkpoints`.

Run all. Expected global top-5: N_RPeaks, HRV_RR_Interval, HRV_CVRR, HRV_SDNN, HRV_RMSSD.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except NameError:
    pass
import importlib, sys, subprocess
for pkg in ("shap", "lightgbm", "scipy"):
    try:
        importlib.import_module(pkg)
    except ImportError:
        try:
            get_ipython().run_line_magic("pip", "install -q " + pkg)
        except NameError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
import sys
print("Python", sys.version.split()[0], "| ready")


In [ ]:
import os, re, ast, json, time, warnings
import numpy as np, pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, confusion_matrix
from scipy.stats import spearmanr
import matplotlib
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

print("NOTEBOOK VERSION: 2.4 (map-fix + §4 Sokolow panel + §8b subset + §8c figures)")

# ================ CONFIG (auto-detection কাজ না করলেই এটা edit করুন) ================
DATA_DIR = "/content/drive/MyDrive/PTB-XL Dataset Analysis"
FEATURES = None   # None = auto-detect। ভুল file picked হলে এখানে exact path দিন
ML_FILE  = None   # None = auto-detect (multi-label 0/1 matrix)
PERMS    = 3      # label permutations (1 = দ্রুত)
N_JOBS   = 2
# =====================================================================================

CLASS_ORDER = ["NORM", "MI", "STTC", "CD", "HYP"]
NON_FEATURE_COLS = {"filename", "record_id", "patient_id", "subject_id", "strat_fold",
                    "label", "Disease_Name", "Disease_Label_Encoded",
                    "Disease_NORM", "Disease_MI", "Disease_STTC", "Disease_CD", "Disease_HYP",
                    "ECG_ID", "ecg_id"}
PARAMS = dict(objective="multiclass", num_class=5, learning_rate=0.05, n_estimators=800,
              num_leaves=63, min_child_samples=40, subsample=0.9, subsample_freq=1,
              colsample_bytree=0.7, reg_lambda=5.0, random_state=42, n_jobs=N_JOBS, verbose=-1)
R = {}

if not os.path.isdir(DATA_DIR):
    raise SystemExit("DATA_DIR not found: " + DATA_DIR +
                     "\n(Colab-এ Drive mounted আছে কিনা দেখুন, আর folder name-এর spelling ঠিক আছে কিনা)")

all_files = []
for root, _, fs in os.walk(DATA_DIR):
    if os.path.basename(root) == "_analysis_checkpoints":
        continue
    for f in fs:
        if f.startswith("."):
            continue
        p = os.path.join(root, f)
        if os.path.isfile(p):
            all_files.append((p, os.path.getsize(p)))
print("=== files in", DATA_DIR, "===")
for p, s in sorted(all_files):
    print(f"  {os.path.relpath(p, DATA_DIR)}   {s/1024/1024:.1f} MB")

for p, s in all_files:
    if p.lower().endswith(".json") and s < 200_000:
        print(f"\n=== {os.path.basename(p)} (audit file) ===")
        print(open(p).read()[:1200])

def find_file(*pats):
    for p, s in all_files:
        n = os.path.basename(p).lower()
        if any(x in n for x in pats):
            return p
    return None

PTBXL_DB = find_file("ptbxl_database")
print("\nPTBXL_DB:", os.path.basename(PTBXL_DB) if PTBXL_DB else "NOT FOUND")

def detect_columns(d):
    cols = list(d.columns)
    def find(*pats):
        for pat in pats:
            for c in cols:
                if pat.lower() in str(c).lower():
                    return c
        return None
    fold = find("strat_fold", "stratfold", "_fold")
    if fold is None:
        for c in cols:
            u = np.unique(d[c].dropna())
            if len(u) == 10 and set(u.tolist()).issubset(set(range(1, 11))):
                fold = c
                break
    patient = find("patient", "subject")
    idc = find("filename", "record_id", "ecg_id", "ecgid", "record id")
    lab = find("label", "diagnosis", "disease", "primary")
    return fold, patient, idc, lab

def n_features(d, fold, patient, lab, idc):
    excl = {fold, patient, lab, idc}
    return len([c for c in d.select_dtypes("number").columns
                if c not in excl and c not in NON_FEATURE_COLS and not str(c).startswith("Disease_")])

cand = []
for p, s in all_files:
    n = os.path.basename(p).lower()
    if n.endswith((".csv", ".parquet", ".pq")) and s > 300_000 \
       and "fqr" not in n and "database" not in n and "metadata" not in n and "scp_" not in n:
        cand.append(p)
def prio(p):
    n = os.path.basename(p).lower()
    sc = 0
    if "model_ready" in n and "stratfold" in n: sc -= 4
    elif "stratfold" in n: sc -= 3
    if "final" in n: sc -= 2
    if "master" in n: sc -= 1
    return sc
cand.sort(key=prio)
print("\n=== candidate feature files (priority order) ===")
for p in cand:
    print("  ", os.path.relpath(p, DATA_DIR))

valid = []
for p in cand:
    name = os.path.basename(p)
    print(f"\n--- {name} ---")
    try:
        d = pd.read_parquet(p) if p.endswith((".parquet", ".pq")) else pd.read_csv(p)
    except Exception as e:
        print("  load failed:", type(e).__name__, str(e)[:120])
        continue
    fold, patient, idc, lab = detect_columns(d)
    probs = [f"no {nm} column" for nm, c in (("fold", fold), ("patient", patient), ("label", lab)) if c is None]
    if not probs:
        fu = set(np.unique(d[fold].dropna()))
        if not (fu.issubset(set(range(1, 11))) or fu.issubset(set(range(10)))):
            probs.append(f"fold values {sorted(fu)[:12]}")
        lv = set(d[lab].astype(str).unique())
        if not (lv.issubset(set(CLASS_ORDER)) or lv.issubset({str(i) for i in range(5)})):
            probs.append(f"label values {sorted(lv)[:8]}")
        dup = d.groupby(patient)[fold].nunique()
        if (dup > 1).any():
            probs.append(f"{int((dup > 1).sum())} patients in >1 fold")
    nf = n_features(d, fold, patient, lab, idc)
    print(f"  rows={len(d)}  feature_cols={nf}  fold={fold}  patient={patient}  id={idc}  label={lab}")
    if lab is not None:
        print("  label counts:", d[lab].value_counts().to_dict())
    print("  problems:", probs or "none")
    if not probs and nf >= 400:
        valid.append([p, d, fold, patient, idc, lab, nf])
        if len(d) == 20310:
            print("  >>> exact cohort size 20,310 — stopping the scan")
            break
    del d

if FEATURES:
    print("\n>>> FEATURES set manually:", FEATURES)
    d = pd.read_parquet(FEATURES) if FEATURES.endswith((".parquet", ".pq")) else pd.read_csv(FEATURES)
    fold, patient, idc, lab = detect_columns(d)
    if not (fold and patient and lab):
        raise SystemExit(f"manual FEATURES given but columns not detected: fold={fold} patient={patient} label={lab} — set them manually below")
    valid = [[FEATURES, d, fold, patient, idc, lab, n_features(d, fold, patient, lab, idc)]]
if not valid:
    raise SystemExit("NO candidate file passed validation. Set FEATURES manually in CONFIG and re-run this cell. "
                     "Send the report above if you are unsure which file to pick.")
valid.sort(key=lambda v: (0 if len(v[1]) == 20310 else 1, -v[6]))
FEATURES, df, FOLD_COL, PATIENT_COL, ID_COL, LABEL_COL, nf = valid[0]
for v in valid[1:]:
    del v[1]
print(f"\n>>> CHOSEN feature file: {os.path.basename(FEATURES)}   (rows={len(df)}, feature cols={nf})")
if len(df) != 20310:
    print(">>> NOTE: rows != 20,310 — verify this is the final cohort file!")

if ID_COL is None:
    raise SystemExit("STOP: record-id column not found (looked for filename / record_id / ecg_id / ecgid). "
                     "Do NOT run the sections — send me the column list printed above.")
if ID_COL == LABEL_COL:
    raise SystemExit("STOP: id detection picked the label column — send me the detection report above.")
print("id column:", ID_COL, "| dtype:", df[ID_COL].dtype, "| sample:", df[ID_COL].head(3).tolist(),
      "| unique:", int(df[ID_COL].nunique()), "/", len(df))
if df[ID_COL].nunique() != len(df):
    print(">>> WARNING: id column is not unique — id-based joins (Sections 4, 7) may be wrong")

fvals = set(np.unique(df[FOLD_COL]))
if fvals.issubset(set(range(10))):
    df[FOLD_COL] = df[FOLD_COL] + 1
    print(">>> fold values were 0-9, shifted to 1-10")

def _align_numeric_labels(y_raw):
    """Match each numeric label code's count to the canonical PTB-XL 5-class sizes
    (unique counts -> unambiguous) so codes align to [NORM,MI,STTC,CD,HYP].
    Self-verifying: any mismatch -> warn + assume order, never silent mislabel."""
    CANON = {"NORM": 9046, "MI": 3890, "STTC": 3107, "CD": 3044, "HYP": 1223}
    count_to_class = {v: k for k, v in CANON.items()}
    vc = pd.Series(y_raw).value_counts().to_dict()
    if len(vc) != 5:
        print(">>> numeric label column has", len(vc), "unique codes (expected 5); assuming order", CLASS_ORDER, "- VERIFY manually!")
        return y_raw
    remap = {}
    for code in sorted(vc):
        if vc[code] in count_to_class:
            remap[code] = CLASS_ORDER.index(count_to_class[vc[code]])
        else:
            print(">>> WARNING: a label code's count matches no canonical PTB-XL size; assuming order", CLASS_ORDER, "- VERIFY!")
            return y_raw
    if set(remap.values()) != set(range(5)):
        print(">>> WARNING: ambiguous count match; assuming order", CLASS_ORDER, "- VERIFY!")
        return y_raw
    if remap == {i: i for i in range(5)}:
        print(">>> numeric labels already in [NORM,MI,STTC,CD,HYP] order")
        return y_raw
    print(">>> REMAPPING numeric labels (file order != [NORM,MI,STTC,CD,HYP]):")
    for c in sorted(remap):
        print(f"      code {c} -> {CLASS_ORDER[remap[c]]}")
    return np.array([remap[c] for c in y_raw], dtype=int)

if df[LABEL_COL].dtype == object:
    y = df[LABEL_COL].map({c: i for i, c in enumerate(CLASS_ORDER)}).to_numpy()
else:
    y = _align_numeric_labels(df[LABEL_COL].to_numpy(dtype=int))

feature_cols = [c for c in df.select_dtypes("number").columns
                if c not in NON_FEATURE_COLS and c not in (FOLD_COL, PATIENT_COL, LABEL_COL, ID_COL)
                and not str(c).startswith("Disease_")]
X = df[feature_cols]
if ID_COL in feature_cols:
    raise SystemExit(f"STOP: {ID_COL} is inside the feature matrix — detection bug, send me the report above")
counts = pd.Series(y).value_counts().reindex(list(range(5)))
print("feature columns:", len(feature_cols), "  <- expect 543")
if len(feature_cols) != 543:
    print(">>> WARNING: feature count is not 543 — check that id / label / Disease_ columns are excluded before trusting any number")
print("class counts (expect 9046/3890/3107/3044/1223):", dict(zip(CLASS_ORDER, counts.astype(int).tolist())))
if counts.tolist() == [9046, 3890, 3107, 3044, 1223]:
    print(">>> LABEL ORDER CHECK: PASS (NORM 9046 / MI 3890 / STTC 3107 / CD 3044 / HYP 1223)")
else:
    print(">>> LABEL ORDER CHECK: MISMATCH — STOP and verify the label column (see REMAPPING line above) before running the sections!")
g = df.groupby(PATIENT_COL)[FOLD_COL].nunique()
assert (g == 1).all(), f"{(g > 1).sum()} patients in >1 fold"
print("patients:", df[PATIENT_COL].nunique(), "— OK")

if ML_FILE is None:
    for p, s in all_files:
        n = os.path.basename(p).lower()
        if not (n.endswith(".csv") and s < 20_000_000):
            continue
        try:
            d2 = pd.read_csv(p)
        except Exception:
            continue
        cc = {str(c).lower(): c for c in d2.columns}
        need = [cc.get(c) for c in ("norm", "mi", "sttc", "cd", "hyp")]
        if None in need:
            continue
        vals = set(np.unique(d2[need].to_numpy()))
        if not vals.issubset({0, 1, 0.0, 1.0}):
            continue
        if len(d2) < 5000:
            continue
        ML_FILE = p
        print(f"ML_FILE auto-picked: {os.path.basename(p)}  rows={len(d2)}  id_col={d2.columns[0]}")
        if len(d2) != 20192:
            print(">>> NOTE: ML matrix rows != 20,192 — confirm it is the CinC 10-fold set, else set ML_FILE = None")
        break
if ML_FILE:
    print("ML_FILE:", os.path.basename(ML_FILE))
elif PTBXL_DB:
    print("ML_FILE: none — Section 7 will derive the multi-label 0/1 ground truth from ptbxl_database.csv (CinC-2020 superclass rule)")
else:
    print("ML_FILE: none and no ptbxl_database.csv — Section 7 will be skipped")

MI_COL = None
for c in df.columns:
    if re.search(r"^mi[_-]?(superclass|ind|flag|code|label)", str(c), re.I):
        MI_COL = c
        break
print("MI_COL:", MI_COL or "(MI co-occurrence will use PTBXL_DB)")

# ---------------- CHECKPOINT / RESUME ----------------
# প্রতিটা section শেষ হলে এখানে save হয়। Runtime মারা গেলে আবার Run all দিলে
# done section গুলো auto-skip হবে — আরম্ভ হবে শেষ checkpoint থেকে।
import shutil as _shutil
CKPT_DIR = os.path.join(DATA_DIR, "_analysis_checkpoints")
try:
    os.makedirs(CKPT_DIR, exist_ok=True)
except Exception:
    CKPT_DIR = "/content"
RERUN = ["4", "7", "8b", "8c"]  # v2.4: database sections + figure regeneration (~45 min total) শুধু database-dependent section গুলো re-compute হবে (~40 min)। পরের run-এর জন্য RERUN = [] করে দিন
FORCE_RERUN = False    # True = পুরো checkpoint ignore করে শুরু থেকে
if FORCE_RERUN and os.path.exists(os.path.join(CKPT_DIR, "state.json")):
    os.remove(os.path.join(CKPT_DIR, "state.json"))
    print("[checkpoint] FORCE_RERUN=True — পুরনো checkpoint মুছে ফেলা হলো")
R, DONE = {}, set()
if os.path.exists(os.path.join(CKPT_DIR, "state.json")):
    _s = json.load(open(os.path.join(CKPT_DIR, "state.json")))
    R, DONE = _s["R"], set(_s["DONE"])
    _r = set(str(x) for x in RERUN)
    DONE -= _r
    print(f"[checkpoint] পুরনো run পাওয়া গেছে — done: {sorted(DONE)} (এই section গুলো skip হবে)")
    if _r:
        print(f"[checkpoint] RERUN={sorted(_r)} — এই section গুলো আবার হিসাব হবে")
else:
    print(f"[checkpoint] নতুন run — প্রতিটা section শেষে save হবে: {CKPT_DIR}")
def _done(sec):
    DONE.add(str(sec))
    for _f in [f for f in os.listdir(".")
                 if (f.startswith("probs_oof_") or f.startswith("figure")) and (f.endswith(".csv") or f.endswith(".png"))]:
        if os.path.exists(_f):
            _shutil.copy(_f, CKPT_DIR)
    json.dump({"R": R, "DONE": sorted(DONE), "ts": time.strftime("%Y-%m-%d %H:%M:%S")},
              open(os.path.join(CKPT_DIR, "state.json"), "w"), default=str)
    print(f"[CHECKPOINT] section {sec} done — saved — এখানে থামা/ভুলে যাওয়া safe")
print("\nLOAD + AUTO-DETECTION DONE")


In [ ]:
def class_sample_weights(y_train, K=5):
    """Eq. 13: w_c = N / (K * N_c), applied per sample (== LightGBM class_weight)."""
    N = len(y_train)
    counts = np.bincount(y_train, minlength=K)
    w = N / (K * np.maximum(counts, 1))
    return w[y_train]

def run_cv(df, X, y, feature_cols, exclude=(), tag=""):
    """Patient-level 10-fold: val fold 9 (or 8 when test fold k == 9), early-stop patience 50."""
    feats = [c for c in feature_cols if c not in set(exclude)]
    folds = df[FOLD_COL].to_numpy()
    n = len(df)
    probs_oof = np.zeros((n, 5))
    t0 = time.time()
    for k in range(1, 11):
        test = np.where(folds == k)[0]
        val = np.where(folds == (8 if k == 9 else 9))[0]
        train = np.setdiff1d(np.arange(n), np.union1d(test, val))
        m = lgb.LGBMClassifier(**PARAMS)
        m.fit(X.loc[train, feats].to_numpy(), y[train], sample_weight=class_sample_weights(y[train]),
              eval_set=[(X.loc[val, feats].to_numpy(), y[val])],
              eval_metric="multi_logloss",
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
        probs_oof[test] = m.predict_proba(X.loc[test, feats].to_numpy())
        try:
            fa = roc_auc_score(y[test], probs_oof[test], multi_class='ovr', average='macro')
            fa_s = f"{fa:.4f}"
        except ValueError:
            fa_s = "n/a (class missing in fold)"
        print(f"  [{tag}] fold {k}: test AUC {fa_s}  (iter {m.best_iteration_})")
    print(f"  [{tag}] done in {(time.time()-t0)/60:.1f} min")
    return probs_oof

def metrics(y, probs):
    pred = probs.argmax(axis=1)
    out = dict(accuracy=float(accuracy_score(y, pred)),
               macro_f1=float(f1_score(y, pred, average="macro")),
               macro_auc=float(roc_auc_score(y, probs, multi_class="ovr", average="macro")),
               per_class_auc={c: float(roc_auc_score((y == i).astype(int), probs[:, i])) for i, c in enumerate(CLASS_ORDER)},
               per_class_f1={c: float(f1_score((y == i).astype(int), pred == i)) for i, c in enumerate(CLASS_ORDER)})
    cm = confusion_matrix(y, pred, labels=list(range(5)))
    out["error_breakdown"] = dict(total_errors=int(cm.sum() - np.trace(cm)),
                                  diseased_to_NORM=int(cm[1:5, 0].sum()),
                                  NORM_to_diseased=int(cm[0, 1:5].sum()),
                                  disease_to_disease=int(cm[1:5, 1:5].sum() - np.trace(cm[1:5, 1:5])),
                                  confusion_matrix=cm.tolist())
    return out

_BUILTIN_SC = {
    # official PTB-XL (v1.0.0-1.0.3) diagnostic statements -> superclass — 44 codes,
    # verified identical to the official scp_statements.csv (physionet ptb-xl/1.0.3)
    "NORM": "NORM",
    "IMI": "MI", "ASMI": "MI", "ILMI": "MI", "AMI": "MI", "ALMI": "MI", "INJAS": "MI",
    "LMI": "MI", "INJAL": "MI", "IPLMI": "MI", "IPMI": "MI", "INJIN": "MI", "INJLA": "MI",
    "PMI": "MI", "INJIL": "MI",
    "NDT": "STTC", "NST_": "STTC", "DIG": "STTC", "LNGQT": "STTC", "ISC_": "STTC",
    "ISCAL": "STTC", "ISCIN": "STTC", "ISCIL": "STTC", "ISCAS": "STTC", "ISCLA": "STTC",
    "ANEUR": "STTC", "EL": "STTC", "ISCAN": "STTC",
    "LAFB": "CD", "IRBBB": "CD", "1AVB": "CD", "IVCD": "CD", "CRBBB": "CD", "CLBBB": "CD",
    "LPFB": "CD", "WPW": "CD", "ILBBB": "CD", "3AVB": "CD", "2AVB": "CD",
    "LVH": "HYP", "LAO/LAE": "HYP", "RVH": "HYP", "RAO/RAE": "HYP", "SEHYP": "HYP",
}

def load_superclass_map():
    """{SCP code: superclass} over the 44 official diagnostic statements.
    Prefers scp_statements.csv (column 'superclass' or 'diagnostic_class'); else built-in
    (which is the official v1.0.3 map itself)."""
    cands = []
    if PTBXL_DB:
        cands.append(os.path.join(os.path.dirname(PTBXL_DB) or ".", "scp_statements.csv"))
    if "DATA_DIR" in globals() and os.path.isdir(DATA_DIR):
        cands.append(os.path.join(DATA_DIR, "scp_statements.csv"))
    for c in cands:
        if os.path.exists(c):
            try:
                s = pd.read_csv(c)
            except Exception:
                print("NOTE: scp_statements.csv unreadable — using built-in map (official 44 codes)")
                continue
            code_col = next((x for x in s.columns if str(x).lower() in ("code", "scp_code")), s.columns[0])
            sc_col = next((x for x in s.columns
                           if "superclass" in str(x).lower() or "diagnostic_class" in str(x).lower()), None)
            if sc_col is None:
                print("NOTE: scp_statements.csv found but no superclass/diagnostic_class column — using built-in map (official 44 codes)")
                continue
            m = {}
            for _, row in s.iterrows():
                if pd.notna(row[sc_col]) and str(row[sc_col]).strip():
                    m[str(row[code_col]).strip()] = str(row[sc_col]).strip().upper()
            if len(m) >= 40:
                from collections import Counter as _C
                print("superclass map: from", os.path.basename(c), f"({len(m)} codes | "
                      + ", ".join(f"{k} {v}" for k, v in sorted(_C(m.values()).items())) + ")")
                return m
            print("NOTE: scp_statements.csv parsed but only", len(m),
                  "diagnostic codes (expected 44) — using built-in map (official 44 codes)")
    print("superclass map: built-in official PTB-XL 44 diagnostic codes (scp_statements.csv not found)")
    return dict(_BUILTIN_SC)

def db_indexed():
    """PTBXL_DB indexed by its id column. Returns (db, scp_codes_column) or None."""
    if not PTBXL_DB:
        return None
    db = pd.read_csv(PTBXL_DB)
    idc = next((c for c in db.columns if str(c).lower() in
                {"filename", "record_id", "ecgid", "ecg_id", "file_name", "id"}), None)
    if idc is None:
        idc = db.columns[0]
        print("NOTE: database id column not obviously named; using first column:", idc)
    scpc = next((c for c in db.columns if "scp" in str(c).lower() and "code" in str(c).lower()), None)
    if scpc is None:
        non_id = [c for c in db.columns if c != idc]
        scpc = non_id[0] if non_id else None
        if scpc is not None:
            print("NOTE: scp_codes column not obviously named; using:", scpc)
    if scpc is None:
        print("WARNING: could not find id/scp_codes columns in", PTBXL_DB, "— columns:", list(db.columns))
        return None
    print("database: id column =", idc, "| scp column =", scpc)
    return db.set_index(idc), scpc

def _codes_to_flags(s, scmap):
    """scp_codes string -> [NORM, MI, STTC, CD, HYP] 0/1 flags (CinC-2020 rule: any statement of the superclass)."""
    try:
        codes = ast.literal_eval(s)
    except Exception:
        return [0, 0, 0, 0, 0]
    flags = [0, 0, 0, 0, 0]
    for c in codes:
        sc = scmap.get(str(c).strip())
        if sc in CLASS_ORDER:
            flags[CLASS_ORDER.index(sc)] = 1
    return flags

def mi_indicator(df):
    """0/1: record carries >= 1 MI superclass statement (from MI_COL or PTB-XL database)."""
    if MI_COL and MI_COL in df.columns:
        return df[MI_COL].fillna(0).astype(int).to_numpy()
    _db = db_indexed()
    if _db is None:
        print("WARNING: no MI_COL / PTBXL_DB available — MI co-occurrence panel will be skipped")
        return None
    dbx, scpc = _db
    scmap = load_superclass_map()
    matched = int(df[ID_COL].isin(dbx.index).sum())
    print(f"MI join: {matched}/{len(df)} feature-file ids found in the database")
    if matched < 0.95 * len(df):
        print("WARNING: id join failed — feature ids sample:", df[ID_COL].head(3).tolist(),
              "| database ids sample:", list(dbx.index[:3]))
    mi = np.array([_codes_to_flags(s, scmap)[1] for s in dbx[scpc].reindex(df[ID_COL])], dtype=int)
    if (mi == 0).all():
        print("WARNING: no record matched an MI statement — check the id columns above")
    return mi
print("HELPERS LOADED")

## Figure 6 — SHAP, fold-10 model

In [ ]:
try:
    import shap
except ImportError:
    import subprocess, sys; subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'shap']); import shap


In [ ]:
# ============================================================================
# 8f) Figure 6 — SHAP (fold-10 model) + export for verification   [v2.4 notebook add-on]
# Paste this as a NEW code cell right after Section 8 (cell "8) SHAP on all 10 folds")
# and run it. Needs the notebook's objects: df, X, y, feature_cols, FOLD_COL, PARAMS,
# class_sample_weights, CKPT_DIR, R, _done.    Runtime: ~3-5 min (one fold + exact SHAP)
# Outputs (saved in _analysis_checkpoints, same Drive folder you shared):
#   Figure6_shap.png, shap_fold10.npy, X_fold10.csv, shap_global_fold10.csv
# ============================================================================
_SEC = "8f"
import shap, numpy as np, pandas as pd, matplotlib.pyplot as plt
CLASS_NAMES = ["NORM", "MI", "STTC", "CD", "HYP"]          # same order as y encoding (0..4)

folds = df[FOLD_COL].to_numpy(); n = len(df); k = 10
test  = np.where(folds == k)[0]
val   = np.where(folds == 9)[0]                              # fold 9 = early-stopping fold when test = 10
train = np.setdiff1d(np.arange(n), np.union1d(test, val))
m = lgb.LGBMClassifier(**PARAMS)
m.fit(X.loc[train, feature_cols].to_numpy(), y[train], sample_weight=class_sample_weights(y[train]),
      eval_set=[(X.loc[val, feature_cols].to_numpy(), y[val])], eval_metric="multi_logloss",
      callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
Xt = X.loc[test, feature_cols].reset_index(drop=True)
sv = shap.TreeExplainer(m).shap_values(Xt.to_numpy())
sv = np.stack(sv, -1) if isinstance(sv, list) else np.asarray(sv)   # n x 543 x 5
assert sv.shape[1] == len(feature_cols) and sv.shape[2] == 5, sv.shape

# ---- exports ----
np.save(os.path.join(CKPT_DIR, "shap_fold10.npy"), sv.astype(np.float32))
Xt.to_csv(os.path.join(CKPT_DIR, "X_fold10.csv"), index=False)
glob = pd.Series(np.abs(sv).mean(axis=(0, 2)), index=feature_cols).sort_values(ascending=False)
glob.to_csv(os.path.join(CKPT_DIR, "shap_global_fold10.csv"), header=["mean_abs_shap"])
print("fold-10 global top-5:", list(glob.index[:5]))
print("  (paper / state.json expects: N_RPeaks, HRV_RR_Interval, HRV_CVRR, HRV_SDNN, HRV_RMSSD)")

# ---- Figure 6: (a) global top-15 bar, (b) NORM beeswarm, (c) MI beeswarm ----
fig = plt.figure(figsize=(12, 15), dpi=200)
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1.35], hspace=0.35, wspace=0.6)
ax0 = fig.add_subplot(gs[0, :]); top = glob.iloc[:15][::-1]
ax0.barh(top.index, top.values, color="#2f6fa8"); ax0.grid(axis="x", alpha=.3)
ax0.set_xlabel("mean |SHAP value| (log-odds), averaged over fold-10 test ECGs and the five classes")
ax0.set_title("(a) Global importance, fold-10 model, top 15 of 543 features", loc="left", fontweight="bold")
for j, cls in enumerate(["NORM", "MI"]):
    ax = fig.add_subplot(gs[1, j]); plt.sca(ax)
    shap.summary_plot(sv[:, :, CLASS_NAMES.index(cls)], Xt, max_display=15, show=False,
                      plot_size=None, color_bar=(j == 1))
    ax.set_title(f"({'bc'[j]}) {cls} class", loc="left", fontweight="bold"); ax.set_xlabel("SHAP value (log-odds)")
out = os.path.join(CKPT_DIR, "Figure6_shap.png")
fig.savefig(out, bbox_inches="tight", facecolor="white"); plt.show()
print("saved:", out)
R["shap_fig6"] = dict(fold10_global_top5=list(glob.index[:5]), n_test=int(len(test)),
                      best_iteration=int(getattr(m, "best_iteration_", 0) or 0))
_done("8f")
print("SECTION 8f DONE")
